# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

In [3]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

In [ ]:
# from uuid import uuid4

# os.environ["LANGCHAIN_PROJECT"] = f"PSI-SDG - {uuid4().hex[0:8]}"

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [4]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [5]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [6]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [7]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [9]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [10]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [18]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided data, the most common issues with loans appear to be related to problems with how loans are handled by lenders or servicers. These issues include:\n\n- Dealing with your lender or servicer (e.g., errors in loan balances, misapplied payments, wrongful denials of payment plans, or incorrect account statuses)\n- Trouble with how payments are being handled or applied (e.g., payments only applied to interest, inability to pay down principal, or payments misapplied)\n- Receiving bad or incorrect information about the loan (e.g., incorrect loan balances, interest rate discrepancies, or status updates)\n- Issues with updating or fixing account status, including delinquency reporting, or misreporting to credit bureaus\n- Cases of privacy violations or legal discrepancies, such as unauthorized disclosure of personal information\n\nOverall, the most prevalent issue seems to be difficulties and errors in proper management and processing of loans by lenders or servicers, ofte

In [11]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, some complaints did not get handled in a timely manner. For example, a complaint received on 03/28/25 by MOHELA was marked as "No" for timely response, indicating it was not handled promptly. Additionally, multiple complaints mention delays of over a month or more before they received a response or resolution.'

In [12]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans mainly due to a combination of factors including:\n- Lack of clear information about repayment obligations and the resumption of payments after forbearance or deferment periods.\n- Increasing interest accumulating during forbearance, which can negate payments and extend payoff timelines.\n- Financial hardships such as stagnant wages, economic downturns, and unexpected expenses making it difficult to afford payments.\n- Poor communication or unexpected notices about loan status or delinquency, leading to unintentional missed payments.\n- Management issues and transfers between loan servicers without proper notification, causing confusion and difficulty in managing repayment.\n- In some cases, borrowers felt misled by the terms or unclear information provided at the outset, leading to struggles with managing or understanding their debt.\n\nOverall, inadequate information, administrative errors, and financial hardship are common reasons that contribu

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [11]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [12]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [13]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with student loans appears to involve handling and dealing with the lender or servicer, particularly in areas such as disputes over fees, applying payments correctly, understanding loan balances and terms, and receiving accurate information. Several complaints highlight problems with loan repayment practices, misinformation, and lack of transparency from loan servicers.'

In [16]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, all the complaints listed in the context were responded to in a timely manner. Each complaint has a "Timely response?" status marked as "Yes." Therefore, there is no indication that any complaints did not get handled in a timely manner.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including problems with their payment plans, miscommunication or lack of communication from loan servicers, being misled into the wrong types of forbearances, and issues with automatic payment enrollment and updates. Some borrowers also experienced their loans being transferred or sold to different companies without proper notification, leading to missed payments or negative impacts on their credit scores. Additionally, difficulties in resolving repayment issues or obtaining assistance through customer service contributed to loan repayment failures.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

</div>

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">
### Answer:

-
- 

</div>

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [14]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [15]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [16]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints, a common issue with loans, particularly student loans, appears to be mishandling by servicers, such as errors in loan balances, misapplied payments, wrongful denials of payment plans, and legal discrepancies or privacy violations. Additionally, a significant concern involves difficulties in managing payments due to high interest accumulation, lack of clear information, and limited options for financial relief, which contribute to borrower hardship.'

In [ ]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided context, it appears that there were complaints that experienced delays in being handled. For example:\n\n- The complaint about the loan account review has been open for over 18 months with no resolution and was still awaiting response.\n- Another complaint concerning violations of federal privacy law involved no response from the company as of the latest update.\n\nAdditionally, multiple complaints indicate that issues remain unresolved for extended periods, sometimes over a year or more, suggesting they were not handled in a timely manner.\n\nTherefore, yes, some complaints did not get handled in a timely manner.'

In [ ]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People often failed to pay back their loans due to a combination of factors including lack of awareness about repayment obligations, miscommunication from loan servicers, inability to keep up with mounting interest, and limited options for manageable repayment plans. Many borrowers were not adequately informed that they needed to repay the loans, and some were unaware of the conditions or changes in their loan servicing, such as transfers without notice. Additionally, options like forbearance or deferment, while accessible, led to continued interest accumulation, which increased the total amount owed and made repayment more difficult over time. Economic hardships, stagnant wages, and the complex interest calculations further contributed to borrowers struggling with repayment, often feeling misled or unsupported by their loan servicers.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [17]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [18]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [19]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints and context, the most common issues with student loans appear to be:\n\n- Dealing with lenders or servicers, including trouble with payment handling, misapplication of payments, or being unable to pay off loans quickly.\n- Errors in loan balances, interest calculations, or account status.\n- Lack of proper communication, notifications, or transparency from loan servicers, including transfers of loan ownership without notice.\n- Problems with loan repayment plans, including being placed in forbearance or deferment, with accruing interest and extended payoff periods.\n- Mismanagement or mishandling in the process of applying for forgiveness or loan discharge.\n- Incorrect or outdated reporting on credit reports, affecting credit scores.\n- Poor customer service, unhelpful representatives, or inability to resolve account errors.\n\nWhile the specific "most common" issue is not explicitly stated as a single problem, the recurring theme across complaints is

In [ ]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, it appears that several complaints involved delays and issues in handling their concerns in a timely manner. Specifically:\n\n- One complaint (Row 423) about a loan application process still unprocessed after over 2-3 weeks.\n- Multiple complaints (Rows 129, 131, 132, and 126) about responses not being received within the promised time frame, with some exceeding the expected response time (e.g., over 15 days, over 1 year, or multiple months).\n- A complaint (Row 130) about credit reporting errors that were not promptly corrected, with a specific note that the issue persisted beyond the promised 48-hour resolution.\n- Other complaints mention wait times of several hours or days to get assistance or corrections, sometimes over weeks or months.\n  \nTherefore, yes, several complaints indicate that issues were not handled in a timely manner.'

In [ ]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'Based on the information provided, people failed to pay back their loans primarily due to factors such as:\n\n- Lack of clear information about repayment options, including income-driven repayment plans and loan rehabilitation, leading many to rely on forbearance or deferment, which can result in accruing interest and increased balances.\n- Accumulation of interest during periods of forbearance or deferment without proper notification, causing loan balances to balloon and making repayment more difficult.\n- Servicer practices such as "forbearance steering" and misinforming borrowers about their repayment options, often coercing them into less advantageous repayment strategies like consolidation, which can reset forgiveness timelines and increase totals.\n- Financial hardships such as unemployment, medical issues, or unexpected expenses, compounded by unmanageable loan balances and high interest, making it difficult to stay current.\n- Poor communication and lack of transparency from l

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

</div>

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

### Answer:

- Generating alternative queries of the initial query broadens the search scope with each query variant potentially identifying different sets of relevant documents. The consolidated list of these documents ideally covers multiple perspectives of the original query thus increasing the likelihood of finding the relevant semantic representations in the vector space, consequently improving recall.

</div>

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [20]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [21]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [22]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [23]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [24]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [25]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided context, appears to be problems related to mismanagement, lack of transparency, or unfair treatment by lenders or servicers. Specific issues mentioned include:\n\n- Struggling to repay loans due to unforeseen or misrepresented terms.\n- Receiving bad or incomplete information about loan terms, interest rates, or payment details.\n- Discrepancies and increases in interest rates that were not originally agreed to.\n- Lack of proper communication about loan consolidation or forgiveness processes.\n- Errors in credit reporting affecting credit scores.\n- Overall systemic breakdowns impacting borrowers nationwide.\n\nAmong these, "dealing with your lender or servicer" issues such as inaccurate information, lack of transparency, and unfair treatment seem to be the most frequently highlighted problems.'

In [ ]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, based on the provided complaints, there are multiple instances where complaints did not get handled in a timely manner. Specifically, at least three complaints indicate delays:\n\n- One complaint received on 03/28/25 from a consumer about a student loan issue, where the company responded with a "No" response to timely handling.\n- Complaints received on 04/11/25, where consumers reported extended wait times (e.g., 4 hours, 7 hours, or more than 7 hours) to speak with representatives, indicating significant delays and ineffective handling.\n- Another complaint submitted on 04/24/25 also indicates a delay—though it was marked "Timely response? Yes," many prior complaints clearly reflect delays and poor response times.\n\nOverall, multiple complaints highlight that issues remained unresolved or responses were significantly delayed, confirming that some complaints did not get handled in a timely manner.'

In [ ]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans primarily due to factors such as experiencing severe financial hardship, difficulties in securing employment after graduation, and misunderstandings or lack of transparency regarding the terms and long-term consequences of their loans. For example, some individuals were misled about the manageability of their student loans and were unaware of the institution's financial instability, which impacted their ability to find suitable jobs and make repayments. Others faced issues like being required to start payments before the end of the grace period or having their payment obligations misunderstood or improperly managed by loan servicers. Additionally, some have encountered problems with loan account management, incorrect reporting, or lack of proper notifications, further complicating their repayment efforts."

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [26]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [27]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [28]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided data, the most common issues with student loan complaints are related to the handling and accuracy of loan information. Specifically, frequent issues include:\n\n- Errors in loan balances and account balances.\n- Misapplication or missing payments.\n- Incorrect or inconsistent reporting of payment history, such as reporting late payments when none occurred, or incorrect delinquency status.\n- Problems with how payments are being applied (e.g., payments not going toward principal).\n- Issues with how loan status and account information are reported to credit bureaus.\n- Problems with loan consolidation, including lack of proper disclosure or unexpected payment amounts.\n- Discrepancies in loan terms and interest calculations.\n\nMany complaints also mention poor communication, unhelpful customer service, and errors arising from transfer of loan servicing between companies, leading to confusion and inaccuracies.\n\n**In summary:**  \nThe most common issue appears t

In [ ]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, several complaints indicate that complaints were not handled in a timely manner. For example:\n\n- Complaint ID 12739706 (Mohela, NJ) received on 04/01/25, where the response was marked "No" for Timely response, and the complaint states they failed to respond within the required timeframe.\n- Complaint ID 12709087 (MOHELA, CA) with a "No" for timely response, indicating delays beyond the expected response time.\n- Complaint ID 12935889 (EdFinancial Services, CA) was also marked "No" for timely response.\n- Complaint ID 12823876 (EdFinancial Services, CA) was timely but involved delays and recurring issues.\n- Complaint ID 12914633 (Higher Education Servicing Corporation, IL) was marked "Yes" for timely response, but the complaint still reflects ongoing issues and delays.\n- Multiple complaints also mention that responses or updates have taken multiple months, or weeks, beyond expected timelines, often with no follow-up or resolution.\n\nIn summar

In [ ]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a combination of factors highlighted in the complaints:\n\n1. **Lack of clear communication and notification:** Borrowers reported not being properly informed about when payments were due, changes in loan servicing companies, or the transfer of their loans. Many were unaware that their payments had resumed or that their loans had been transferred, leading to missed payments and credit issues.\n\n2. **Inadequate guidance and support:** Repeated complaints mention unhelpful or dismissive customer service, failure to provide guidance on alternative repayment plans (like income-driven options), or assistance with deferment and forbearance. This left borrowers unclear about their repayment options and the consequences of missed payments.\n\n3. **Problems with loan servicing practices:** Several complaints allege "forbearance steering," where borrowers are pushed into forbearance rather than being informed about other options that could

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [ ]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [ ]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [ ]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [ ]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [ ]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issues with loans tend to revolve around problems with servicing and administrative errors. Specific common issues include:\n\n- Struggling to repay or problems with loan forgiveness/discharge.\n- Improper or unauthorized reporting of loan information, including credit reporting errors.\n- Trouble with how payments are being handled, such as auto-debit issues or inaccurate payment amounts.\n- Miscommunication or lack of transparency from loan servicers.\n- Disputes over account status, default, or misinformation on credit reports.\n- Unauthorized data breaches or privacy violations.\n\nWhile the data highlights various specific problems, the overall pattern suggests that administrative and servicing issues—such as miscommunication, reporting errors, and difficulties with repayment management—are among the most frequent problems reported with loans.\n\nIf you are looking for a single most common issue, it appears that "problems with loan s

In [ ]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, according to the provided data, several complaints indicate that they were handled in a timely manner, as evidenced by the "Timely response?" field being marked "Yes" for each case. All the complaints listed, including those against Nelnet, Maximus Federal Services, Inc., and EdFinancial Services, were responded to within the expected timeframe, with company responses marked as "Closed with explanation" and no reports of delays or mishandling.\n\nHowever, this information only reflects the response time from the companies and does not necessarily mean all issues were resolved satisfactorily or that some complaints may have had underlying issues unresolved. Based on the available data, there are no complaints explicitly noted as not being handled in a timely manner.\n\nIf you need more detailed analysis or specific cases, please let me know!'

In [ ]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for various reasons, including:\n\n- Lack of transparency and accountability from lenders or servicers, making it difficult to understand the status of their loans and the repayment process.\n- Problems with the handling and verification of documentation related to loan forgiveness or discharge, leading to delays or rejections.\n- Disputes over the legitimacy of debt or reporting errors, which can hinder repayment efforts.\n- Difficulties in accessing accurate and timely information about loans or payments, including technical issues logging into accounts or communication breakdowns.\n- Concerns over illegal or improper practices by loan servicers, such as unauthorized reporting, data breaches, or delays in re-amortization after forbearance.\n- Personal or financial hardships not explicitly mentioned in the complaints but implied by stress or inability to manage increased payments.\n\nIn essence, issues related to miscommunication, administrative 

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

</div>

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

### Answer:

Because semantic similarity drives chunk boundaries, similar and repetitive FAQs may merge into large chunks and depending on the threshold, unrelated FAQs may be lumped together. On the other hand, minor phrasing or word differences in short FAQs may increase embedding distances enough to fall above the threshold, leading to unnecessary context fragmentation. Moreover, since the thresholds depends on the overall distribution of the data, adding or removing FAQs may shift the threshold, making the chunking behavior unpredictable. 

To make semantic chunking perform better for short and repetitive FAQs, the following adjustments to the current code are worth exploring (implementing all may not be necessary):
- add size constraints e.g. minimum chunk size to prevent trivial, context-less chunks and/or maximum chunk size to prevent unrelated FAQs from merging due to high similarity; 
- force splits at known boundaries or structural cues regardless of embedding similarity;
- use semantic chunking inside each heading-defined section; and/or 
- set thresholds based on an absolute similarity drop instead of relative values.


</div>

# 🤝 Breakout Room Part #2

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against each other. 
You can use the loans or bills dataset.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

</div>

##### HINTS:

- LangSmith provides detailed information about latency and cost.

### 🏗️ Activity #1 Implementation

Run the SDG notebook (modified from day 4) to generate the synthetic bills dataset and to upload it to LangSmith.

##### Data Ingestion

In [29]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "bills/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
rag_documents = loader.load()

In [30]:
rag_documents

[Document(metadata={'producer': '', 'creator': 'Image Capture Plus', 'creationdate': '2025-07-02T09:41:16+00:00', 'source': 'bills\\20250725 SBN 25 AI Regulation Act.pdf', 'file_path': 'bills\\20250725 SBN 25 AI Regulation Act.pdf', 'total_pages': 16, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-07-02T09:41:16+00:00', 'trapped': '', 'modDate': 'D:20250702094116Z', 'creationDate': 'D:20250702094116Z', 'page': 0}, page_content="TWENTIETH CONGRESS OF THE \nREPUBLIC OF THE PHILIPPINES \nFirst Regular Session\n25 ,JUL-2 P4 55\nSENATE\nS. No.\n25\nIntroduced by Senator PIA S. CAYETANO\nAN ACT\nREGULATING THE DEVELOPMENT AND USE OF ARTIFICIAL INTELLIGENCE \nSYSTEMS IN THE PHILIPPINES, PROMOTING ETHICAL AND RESPONSIBLE \nARTIFICIAL INTELLIGENCE INNOVATION, AND INTEGRATING \nSUSTAINABILITY AND FUTURES THINKING IN NATIONAL POLICY MAKING, \nAND FOR OTHER PURPOSES\nEXPLANATORY NOTE\nThe rise of Artificial Intelligence (AI) is profoundly transformi

In [31]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

In [32]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [33]:
from langchain_community.vectorstores import Qdrant

rag_vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loans RAG"
)

##### Retrievers

In [37]:
naive_retriever = rag_vectorstore.as_retriever(search_kwargs={"k" : 10})

bm25_retriever = BM25Retriever.from_documents(rag_documents, ) 

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [38]:
parent_docs = rag_documents
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)


client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="bills_full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="bills_full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)


bills_store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=bills_store,
    child_splitter=child_splitter,
)

parent_document_retriever.add_documents(parent_docs, ids=None)

In [39]:
retriever_list = [naive_retriever, bm25_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

In [40]:
loans_all_retrievers = {
    "naive": naive_retriever, 
    "bm25": bm25_retriever, 
    "parent-document": parent_document_retriever, 
    "reranking": compression_retriever, 
    "multi-query": multi_query_retriever, 
    "ensemble": ensemble_retriever
}

##### Ragas Evaluation

In [41]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    answer_correctness,
    context_recall,
    context_precision,
)

metrics = [
    context_recall,
    context_precision,
    # faithfulness,
    # answer_relevancy,
    # answer_correctness,
]

In [42]:
import pandas as pd
import nest_asyncio

nest_asyncio.apply()
test_df = pd.read_csv('bills/sdg_bills.csv')

In [44]:
test_df = test_df[:10]
test_df

,user_input,reference_contexts,reference,synthesizer_name
0,Who is PIA S. CAYETANO in the context of AI re...,"[""TWENTIETH CONGRESS OF THE \nREPUBLIC OF THE ...",PIA S. CAYETANO is the senator who introduced ...,single_hop_specifc_query_synthesizer
1,What is Georgetown University known for?,['AI presents enormous opportunities for the P...,The context mentions Georgetown University as ...,single_hop_specifc_query_synthesizer
2,What role did the TWENTIETH CONGRESS of the Ph...,['TWENTIETH CONGRESS OF THE \nREPUBLIC OF THE ...,"The TWENTIETH CONGRESS of the Philippines, thr...",single_hop_specifc_query_synthesizer
3,What is AGI in the context of AI policies in t...,"['1 \na) Promote innovation, technological adv...",The context states that the Act shall regulate...,single_hop_specifc_query_synthesizer
4,How does skills mapping and future-proofing re...,"[""<1-hop>\n\n1 \niii) \nProof of employer enga...",Skills mapping and future-proofing involve con...,multi_hop_abstract_query_synthesizer
5,"H0w do the licenss, certifcations, and penalti...",['<1-hop>\n\n1 \nSec. 12. Contents of AI Regis...,The context states that the registration of AI...,multi_hop_abstract_query_synthesizer
6,how does the ai oversight impact assessments a...,"[""<1-hop>\n\n1 \nSec. 15. AI Ethics Review Boa...",the ai oversight impact assessments are part o...,multi_hop_abstract_query_synthesizer
7,How do penalties for deceitful endorsements an...,"[""<1-hop>\n\n1 \nendorsements, voice recording...",The context states that endorsements and voice...,multi_hop_abstract_query_synthesizer
8,how much Php 2000000 or Php 1000000 fines for ...,['<1-hop>\n\n1 \ndevelopment priorities. These...,the context says if you develop or use AI with...,multi_hop_specific_query_synthesizer
9,how does DOLE relate to AI regulation and harm...,['<1-hop>\n\n1 \nSec. 6. Jurisdiction of the N...,"The NAIC has jurisdiction over AI matters, inc...",multi_hop_specific_query_synthesizer


In [45]:
test_questions = test_df["user_input"].values.tolist()
test_groundtruths = test_df["reference"].values.tolist()

In [46]:
from datasets import Dataset
import nest_asyncio

nest_asyncio.apply()

response_datasets = {}
results = {}

In [47]:
for ret_name, ret in loans_all_retrievers.items():

    os.environ["LANGCHAIN_PROJECT"] = f"PSI-D5-{ret_name}"

    ret_chain = (
        {"context": itemgetter("question") | ret, "question": itemgetter("question")}
        | RunnablePassthrough()
        | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
    )
    answers = []
    contexts = []
    for query in test_questions:
        response = ret_chain.invoke({"question" : query})
        answers.append(response["response"].content)
        contexts.append([context.page_content for context in response["context"]])
    
    res_dataset = Dataset.from_dict({
        "question" : test_questions,
        "answer" : answers,
        "contexts" : contexts,
        "ground_truth" : test_groundtruths
    })

    response_datasets[ret_name] = res_dataset

    results[ret_name] = evaluate(res_dataset, metrics)


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

In [48]:
results

{'naive': {'context_recall': 0.6667, 'context_precision': 0.7999},
 'bm25': {'context_recall': 0.3833, 'context_precision': 0.6000},
 'parent-document': {'context_recall': 0.6667, 'context_precision': 0.7917},
 'reranking': {'context_recall': 0.5500, 'context_precision': 0.8333},
 'multi-query': {'context_recall': 0.7833, 'context_precision': 0.7448},
 'ensemble': {'context_recall': 0.7667, 'context_precision': 0.7822}}

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

### Analysis & Observations:

</div>